# S17 — Transformer II

**Module 3**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Boyu-Zhang-UOI/dl-f2026-notebooks/blob/main/s17_transformer_ii.ipynb)

Every cell below is a worked example from the [S17 reading](https://boyu-zhang-uoi.github.io/dl-f2026/readings/sessions/s17/) — same code, same seeds, same outputs. Run them, then change things and see what breaks: that is what this notebook is for.

Slides for this session: [s17.html](https://boyu-zhang-uoi.github.io/dl-f2026/slides/s17.html)


In [ ]:
# Colab only: install PyTorch if it is missing (local runs already have it).
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("torch") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch"], check=True)
print("environment ready")

## The complete block, in one page


*Expected output starts with:* `in (2, 8, 64) -> out (2, 8, 64)   (a block preserves its shape)`


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

class Block(nn.Module):
    """One pre-norm Transformer block: x + Attn(LN(x)), then x + FFN(LN(x))."""

    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads, self.d_head = n_heads, d_model // n_heads
        self.ln1, self.ln2 = nn.LayerNorm(d_model), nn.LayerNorm(d_model)
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.proj = nn.Linear(d_model, d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))

    def attn(self, x):
        B, T, D = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        shape = lambda t: t.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        q, k, v = shape(q), shape(k), shape(v)
        scores = q @ k.transpose(-2, -1) / math.sqrt(self.d_head)
        mask = torch.triu(torch.ones(T, T, dtype=torch.bool), diagonal=1)
        w = F.softmax(scores.masked_fill(mask, float("-inf")), dim=-1)
        out = (w @ v).transpose(1, 2).contiguous().view(B, T, D)
        return self.proj(out)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))     # sublayer 1: mix ACROSS positions
        x = x + self.ffn(self.ln2(x))      # sublayer 2: transform WITHIN one
        return x

B, T, d_model, n_heads, d_ff = 2, 8, 64, 4, 256
block = Block(d_model, n_heads, d_ff)
x = torch.randn(B, T, d_model)
y = block(x)

print(f"in {tuple(x.shape)} -> out {tuple(y.shape)}   (a block preserves its shape)")
print()
print(f"{'tensor':<24}{'shape':<22}{'note'}")
rows = [
    ("x (input)",              (B, T, d_model),                   "batch, positions, features"),
    ("qkv(LN(x))",             (B, T, 3 * d_model),               "Q, K, V in one matmul"),
    ("q per head",             (B, n_heads, T, d_model//n_heads), "split d_model across heads"),
    ("scores = q @ k^T",       (B, n_heads, T, T),                "every position vs every position"),
    ("attn @ v",               (B, n_heads, T, d_model//n_heads), "weighted mix of values"),
    ("concat heads -> proj",   (B, T, d_model),                   "back to model width"),
    ("ffn hidden",             (B, T, d_ff),                      "expand (d_ff = 4 * d_model)"),
    ("y (output)",             (B, T, d_model),                   "same shape as x -> stackable"),
]
for name, shape, note in rows:
    print(f"{name:<24}{str(tuple(shape)):<22}{note}")

total = sum(p.numel() for p in block.parameters())
attn_p = sum(q.numel() for m in (block.qkv, block.proj) for q in m.parameters())
ffn_p = sum(q.numel() for q in block.ffn.parameters())
print()
print(f"parameters: {total} total | attention {attn_p} | FFN {ffn_p} "
      f"({100 * ffn_p / total:.0f}% in the FFN)")

## Byte-pair encoding: learn the merges


*Expected output starts with:* `merge 1: 'e' + 's' -> 'es'  (count 9)`


In [ ]:
from collections import Counter

# Byte-pair encoding on a toy corpus (word -> count). Each word starts as
# characters plus an end-of-word marker; we repeatedly merge the most
# frequent adjacent pair.
corpus = {"low": 5, "lower": 2, "newest": 6, "widest": 3}
words = {tuple(w) + ("</w>",): c for w, c in corpus.items()}

def pair_counts(words):
    counts = Counter()
    for symbols, c in words.items():
        for pair in zip(symbols, symbols[1:]):
            counts[pair] += c
    return counts

def apply_merge(words, pair):
    merged = {}
    a, b = pair
    for symbols, c in words.items():
        out, i = [], 0
        while i < len(symbols):
            if i + 1 < len(symbols) and symbols[i] == a and symbols[i + 1] == b:
                out.append(a + b); i += 2
            else:
                out.append(symbols[i]); i += 1
        merged[tuple(out)] = c
    return merged

merges = []
for step in range(8):
    counts = pair_counts(words)
    best, freq = counts.most_common(1)[0]
    merges.append(best)
    words = apply_merge(words, best)
    print(f"merge {step + 1}: {best[0]!r} + {best[1]!r} -> {best[0] + best[1]!r}  (count {freq})")

print("\nvocabulary entries for each corpus word after 8 merges:")
for symbols, c in words.items():
    print(f"  {' '.join(symbols)}")

# Segment an UNSEEN word with the learned merge list (apply in learned order).
def segment(word, merges):
    symbols = tuple(word) + ("</w>",)
    for pair in merges:
        symbols = tuple(next(iter(apply_merge({symbols: 1}, pair))))
    return symbols

for w in ["lowest", "newer", "lowlow"]:
    print(f"unseen word {w!r} -> {' '.join(segment(w, merges))}")

## The trade-off, measured


*Expected output starts with:* `merges  table size  corpus tokens           held-out word split`


In [ ]:
from collections import Counter

# The vocabulary-size trade-off: characters vs learned BPE merges vs words.
text = ("the tokenizer tokenizes the text the transformer transforms "
        "the tokens a learner learns the learned tokens")
corpus = Counter(text.split())
held_out = "learners"   # never appears in the corpus

words = {tuple(w) + ("</w>",): c for w, c in corpus.items()}

def pair_counts(words):
    counts = Counter()
    for symbols, c in words.items():
        for pair in zip(symbols, symbols[1:]):
            counts[pair] += c
    return counts

def apply_merge(words, pair):
    merged = {}
    a, b = pair
    for symbols, c in words.items():
        out, i = [], 0
        while i < len(symbols):
            if i + 1 < len(symbols) and symbols[i] == a and symbols[i + 1] == b:
                out.append(a + b); i += 2
            else:
                out.append(symbols[i]); i += 1
        merged[tuple(out)] = c
    return merged

def segment(word, merges):
    symbols = tuple(word) + ("</w>",)
    for pair in merges:
        symbols = tuple(next(iter(apply_merge({symbols: 1}, pair))))
    return symbols

merges = []
snapshots = {0: dict(words)}
for step in range(1, 31):
    best, _ = pair_counts(words).most_common(1)[0]
    merges.append(best)
    words = apply_merge(words, best)
    if step in (10, 20, 30):
        snapshots[step] = dict(words)

n_base = len({ch for w in corpus for ch in w} | {"</w>"})
print(f"{'merges':>6}  {'table size':>10}  {'corpus tokens':>13}  {'held-out word split':>28}")
for n, snap in snapshots.items():
    total = sum(len(symbols) * c for symbols, c in snap.items())
    seg = segment(held_out, merges[:n])
    # embedding-table size = base symbols + one new token per merge
    print(f"{n:>6}  {n_base + n:>10}  {total:>13}  {' '.join(seg):>28}")

# Word-level for comparison: smallest sequences, but the held-out word
# has no id at all -- it becomes a single unknown token.
n_word_tokens = sum(corpus.values())
print(f"{'words':>6}  {len(corpus) + 1:>10}  {n_word_tokens:>13}  {'[UNK]':>28}")

## Same corpus, different algorithms: WordPiece and unigram


*Expected output starts with:* `step    BPE (by count)   WordPiece (by score)`


In [ ]:
from collections import Counter

# BPE vs WordPiece on the same corpus. Both learn merges greedily; the
# difference is the selection rule. BPE merges the most FREQUENT adjacent
# pair. WordPiece merges the pair with the highest ASSOCIATION score
#   score(a, b) = count(a, b) / (count(a) * count(b))
# -- the pair that co-occurs more than its parts' frequencies predict,
# which is (up to a log) the likelihood gain of adding the merged token
# to a unigram model.

corpus = {"low": 5, "lower": 2, "newest": 6, "widest": 3}

def pair_counts(words):
    counts = Counter()
    for symbols, c in words.items():
        for pair in zip(symbols, symbols[1:]):
            counts[pair] += c
    return counts

def unit_counts(words):
    counts = Counter()
    for symbols, c in words.items():
        for s in symbols:
            counts[s] += c
    return counts

def apply_merge(words, pair):
    merged = {}
    a, b = pair
    for symbols, c in words.items():
        out, i = [], 0
        while i < len(symbols):
            if i + 1 < len(symbols) and symbols[i] == a and symbols[i + 1] == b:
                out.append(a + b); i += 2
            else:
                out.append(symbols[i]); i += 1
        merged[tuple(out)] = c
    return merged

def train(rule, n_merges=6):
    words = {tuple(w) + ("</w>",): c for w, c in corpus.items()}
    picked = []
    for _ in range(n_merges):
        pc, uc = pair_counts(words), unit_counts(words)
        if rule == "bpe":
            best = max(sorted(pc), key=lambda p: pc[p])
        else:  # wordpiece
            best = max(sorted(pc), key=lambda p: pc[p] / (uc[p[0]] * uc[p[1]]))
        picked.append(best)
        words = apply_merge(words, best)
    return picked

bpe, wp = train("bpe"), train("wordpiece")
print(f"{'step':>4}  {'BPE (by count)':>16}  {'WordPiece (by score)':>21}")
for i, (b, w) in enumerate(zip(bpe, wp), 1):
    print(f"{i:>4}  {b[0] + '+' + b[1]:>16}  {w[0] + '+' + w[1]:>21}")

*Expected output starts with:* `'lowest' -> low est   (total log-prob -3.978)`


In [ ]:
import math

# The unigram tokenizer's inference step. Unlike BPE/WordPiece, a unigram
# model does not replay merges. It holds a vocabulary of tokens with
# probabilities and segments new text into the sequence of in-vocabulary
# tokens with the highest total log-probability, found by dynamic
# programming (Viterbi). Here the vocabulary and counts are given; a real
# trainer would also learn them (by EM, pruning a large seed vocabulary).

vocab_counts = {
    "l": 2, "o": 2, "w": 4, "e": 5, "s": 2, "t": 2, "n": 1, "r": 2,
    "lo": 3, "low": 7, "est": 9, "er": 4, "new": 6, "wid": 3, "id": 3,
    "west": 2, "lowe": 1,
}
total = sum(vocab_counts.values())
logp = {tok: math.log(c / total) for tok, c in vocab_counts.items()}

def viterbi_segment(word):
    # best[i] = (score, segmentation) for the prefix word[:i]
    best = [(0.0, [])] + [(-math.inf, None)] * len(word)
    for i in range(1, len(word) + 1):
        for j in range(max(0, i - 5), i):        # max token length 5
            piece = word[j:i]
            if piece in logp and best[j][0] > -math.inf:
                score = best[j][0] + logp[piece]
                if score > best[i][0]:
                    best[i] = (score, best[j][1] + [piece])
    return best[-1]

for word in ["lowest", "newest", "lowers"]:
    score, seg = viterbi_segment(word)
    if seg is None:
        print(f"{word!r}: no segmentation with this vocabulary")
    else:
        print(f"{word!r} -> {' '.join(seg)}   (total log-prob {score:.3f})")

# Why does 'lowest' become low+est and not, say, lo+west or lowe+st?
# Enumerate a few candidate segmentations and score them by hand:
cands = [["low", "est"], ["lo", "west"], ["l", "o", "w", "est"]]
print("\ncandidate scores for 'lowest':")
for seg in cands:
    s = sum(logp[p] for p in seg)
    print(f"  {' + '.join(seg):<22} log-prob = {s:.3f}")

## When tokenizers meet the real world


*Expected output starts with:* `          case  tokens  chars  tok/char   worst word`


In [ ]:
from collections import Counter

# Tokenization pathologies, demonstrated on the toy BPE from earlier in
# this section (retrained here so the script is self-contained). The
# tokenizer is trained on lowercase English prose -- then asked to encode
# capitalized words, digits, and German. Watch the token counts.

text = ("the tokenizer tokenizes the text the transformer transforms "
        "the tokens a learner learns the learned tokens")
corpus = Counter(text.split())
words = {tuple(w) + ("</w>",): c for w, c in corpus.items()}

def pair_counts(words):
    counts = Counter()
    for symbols, c in words.items():
        for pair in zip(symbols, symbols[1:]):
            counts[pair] += c
    return counts

def apply_merge(words, pair):
    merged = {}
    a, b = pair
    for symbols, c in words.items():
        out, i = [], 0
        while i < len(symbols):
            if i + 1 < len(symbols) and symbols[i] == a and symbols[i + 1] == b:
                out.append(a + b); i += 2
            else:
                out.append(symbols[i]); i += 1
        merged[tuple(out)] = c
    return merged

merges = []
for _ in range(30):
    best, _ = pair_counts(words).most_common(1)[0]
    merges.append(best)
    words = apply_merge(words, best)

base = {ch for w in corpus for ch in w} | {"</w>"}
vocab = base | {a + b for a, b in merges}

def segment(word, merges):
    symbols = tuple(word) + ("</w>",)
    for pair in merges:
        symbols = tuple(next(iter(apply_merge({symbols: 1}, pair))))
    return symbols

tests = [
    ("in-domain     ", "the learner learns"),
    ("capitalized   ", "The Learner Learns"),
    ("numbers       ", "learner 2026 44444"),
    ("German        ", "der lernende lernt"),
]
print(f"{'case':>14}  {'tokens':>6}  {'chars':>5}  {'tok/char':>8}   worst word")
for name, sent in tests:
    toks, worst = [], ("", 0)
    for w in sent.split():
        seg = segment(w, merges)
        seg = tuple(s if s in vocab else f"<?{s}?>" for s in seg)  # true OOV symbol
        toks.extend(seg)
        if len(seg) > worst[1]:
            worst = (f"{w} -> {' '.join(seg)}", len(seg))
    n_chars = len(sent.replace(" ", ""))
    print(f"{name:>14}  {len(toks):>6}  {n_chars:>5}  {len(toks)/n_chars:>8.2f}   {worst[0]}")

## Try it yourself

1. In the first script, continue to 12 merges. Predict which pairs win merges 9–12 before running, using the corpus counts.
2. Add `{"slowest": 4}` to the toy corpus and retrain. How do the learned merges change, and how does the unseen word "slower" now segment?
3. In the trade-off script, sweep merges from 0 to 60 in steps of 5 and record table size and corpus tokens for each. Where does compression visibly saturate for this corpus, and why must it saturate?
4. The end-of-word marker looks like a detail. Remove `</w>` from both scripts and rerun: find a concrete pair of strings that becomes indistinguishable, and explain what that would do to a language model's outputs.
5. In the WordPiece comparison, compute by hand the association scores of `e+s` and `i+d` at step 1 from the corpus counts, and confirm they explain the two algorithms' different first merges. Then predict what happens to WordPiece's ordering if you add `{"id": 20}` (the word "id") to the corpus, and check.
6. In the unigram script, double the count of `west` to 4 and rerun. Which segmentations flip, at what count would `lo + west` overtake `low + est` for "lowest", and what does this tell you about how vocabulary probabilities steer segmentation?
7. Extend the pathologies script with a fifth test case of your own design that gets a *worse* tokens-per-character ratio than the German row, and explain the mechanism that makes it expensive.


---

Full discussion of everything above: [S17 reading](https://boyu-zhang-uoi.github.io/dl-f2026/readings/sessions/s17/).
